# 01. Paths

**Tier:** Intermediate. **Why you'd use this:** You need the suffix and parent of a file path without hand-splitting strings that break on every OS. `pathlib.Path` is the object for that.

**After this lesson you can:** build a `Path`; read `.name`, `.suffix`, `.parent`; open files with `with` and an explicit encoding.
About 25 minutes. **Uses:** [Context managers](../12-data-model/02-context-managers-lesson.ipynb)


## Quick review

1. Why does a `__main__` guard matter for importable scripts?
2. Where are installed packages found for `import`?

<details><summary>Answers</summary>

So demos do not run on import. On `sys.path`, including the environment.
</details>


## The idea

`Path` values represent filesystem paths as objects. Parts like `.suffix` and `.parent` are attributes, not brittle string slices. Open text with `with path.open(encoding="utf-8")` so the file closes and the encoding stays explicit.

<details><summary>Explain more</summary>

`Path("notes.txt").suffix` is `".txt"`. `with` guarantees close and flush even when the body errors.
</details>


## See it work

Inspect a path and write a short file.


In [1]:
from pathlib import Path

path = Path("notes.txt")
# 1. Parts without string splitting.
print(path.name)
print(path.suffix)
print(path.parent)

with path.open("w", encoding="utf-8") as handle:
    handle.write("hello\n")
print(path.read_text(encoding="utf-8"))
path.unlink()

notes.txt
.txt
.
hello



## Check yourself

Answer these before you run the cell.

1. Why pass `encoding="utf-8"` even when the platform default might match?
2. What still runs if the body of `with path.open(...)` raises?


In [2]:
from pathlib import Path

path = Path("demo.txt")
with path.open("w", encoding="utf-8") as handle:
    handle.write("ok")
print(path.read_text(encoding="utf-8"))
path.unlink()
print(Path("a/b/c.txt").suffix)

ok
.txt


<details><summary>Answers</summary>

So the file meaning does not depend on the machine default. `__exit__` still closes the file.
</details>


## Why it works this way

Paths are not just strings. Different operating systems use different separators and rules. `pathlib` hides that behind one API so `.suffix` and `/` joining behave consistently. Opening a text file still needs an encoding: bytes on disk become `str` only through a codec. Writing `encoding="utf-8"` documents the contract and keeps behaviour stable across machines, even when defaults improve. Deep dive related skill: [Context managers](../12-data-model/02-context-managers-lesson.ipynb).

`with` matters because files buffer and hold OS resources. `__exit__` closes the handle and flushes buffered data. An exception in the body still runs that cleanup. Forgetting `with` and relying on garbage collection delays close and can truncate writes on some platforms.

Cost of `Path` operations is small string work until you touch the filesystem. Open/read/write dominate. Prefer `Path` over manual joins with `+` and `"/"`. Prefer explicit encoding on every text open.

Evidence: `.suffix` and `.parent` on a path, and a round-trip write/read under `with` with `encoding="utf-8"`. If you omit encoding, you inherit a platform default that is not part of your program's meaning.

Use `read_text`/`write_text` for short whole-file work. Use an open handle when you stream. Delete temp demo files you create in lessons so the folder stays clean.  Path objects are immutable-ish values you can pass around without surprising string mutations. Join with `/` instead of guessing separators. When a path comes from user input, still set encoding yourself on open.

Relative paths resolve against the process working directory, not against the notebook file's folder, unless you build from `__file__` in a script. That distinction causes many "file not found" bugs in notebooks. Prefer absolute paths in tests via fixtures such as `tmp_path`. Document whether a function expects a `Path` or a `str`, and accept both by wrapping with `Path(...)` at the edge.


In [3]:
from pathlib import Path

path = Path("notes.txt")
print(path.name, path.suffix, path.parent)
with path.open("w", encoding="utf-8") as handle:
    handle.write("hello\n")
print(path.read_text(encoding="utf-8"))
path.unlink()
print(Path("dir") / "file.txt")

notes.txt .txt .
hello

dir/file.txt


## When to use it

Use `Path` for all path manipulation. Use `with` and explicit encoding for text IO. Use bytes mode only when you mean bytes. Don't build paths with string concatenation across OSes. Don't leave files open across unrelated code. Keep path handling next to the open call so encoding choices stay visible.


## Common mistake

Opening text without an encoding and assuming every machine agrees.


In [4]:
from pathlib import Path

path = Path("demo.txt")
path.write_text("café")
print(path.read_text())
path.unlink()

café


Pass `encoding="utf-8"` on write and read so the bytes mean the same everywhere.


In [5]:
from pathlib import Path

path = Path("demo.txt")
path.write_text("café", encoding="utf-8")
print(path.read_text(encoding="utf-8"))
path.unlink()

café


## Try it

Build `Path("data") / "items.csv"` and print `.name` and `.suffix`. Which part changes if you use a Windows-style string with backslashes on another OS?


## Recap

- `Path` exposes `.name`, `.suffix`, `.parent`.
- Open text with `with` and `encoding="utf-8"`.
- `with` closes and flushes even after errors.

**Next:** [practice](01-paths-practice.ipynb)
